# Seasonal what-if response by subregion

In [ ]:
# general
from pathlib import Path
import numpy as np
import pandas as pd

# spatial
import xarray as xr
import geopandas as gpd

# plotting
import plotly.graph_objects as go
from   plotly.subplots import make_subplots

from src.utils import load_config, assign_region
from src.plotting import base_layout, save_figure, FONT_FAMILY, MUTED_TEXT_COLOR, POS_COLOR, NEG_COLOR
cfg = load_config()

SCENARIO      = "28C"  # warming level shown in the figure
scenario_name = {s["id"]: s["name"] for s in cfg["scenarios"]}[SCENARIO].split(" (")[0]  # drop "(N=...)"
regions        = list(cfg["regions"])

rows_order  = regions

## Data

In [ ]:
# target basins: subregion from the centroid latitude
basins = gpd.read_file("data/hydro/basins_hydroATLAS_lev08.gpkg").set_index("gauge_id")
basins["region"] = basins["basin_cenlat"].apply(assign_region, regions=cfg["regions"])

In [ ]:
def monthly_melt_by_region(path):
    """Mean monthly glacier melt (m3 s-1) per subregion."""
    da  = xr.open_dataset(path, engine="h5netcdf").glacier_melt
    ids = pd.Index(da.gauge_id.values.astype(str))
    da  = da.sel(gauge_id=ids[ids.str.match(r"^\d")])  # target basins only (drop Andean-GC gauges)

    melt = da.groupby("date.month").mean().to_pandas()  # gauge x month
    meta = basins.loc[melt.index, "region"]

    out = melt.groupby(meta).sum().T
    return out[rows_order]  # month x region

melt_hist = monthly_melt_by_region("data/glacier/run_output_historical.nc")

# what-if runs: ensemble mean of the monthly melt for the selected warming level
files   = sorted(Path("data/glacier/").glob(f"run_output_historical_{SCENARIO}_*.nc"))
members = [monthly_melt_by_region(f) for f in files]
melt_whatif = sum(members) / len(members)
melt_change = 100 * (melt_whatif / melt_hist - 1)

In [ ]:
# drought area (km2) per 1º latitude band; band c holds basins with centroid latitude in (c-1, c]
drought = pd.read_csv("data/hydro/droughts_hydroatlas_area_ts.csv", index_col=["scenario", "date"], parse_dates=["date"])
band_region = pd.Series({c: assign_region(int(c) - 0.5, cfg["regions"]) for c in drought.columns})

drought_reg = drought.T.groupby(band_region).sum().T

# monthly climatology of the area in drought, then relative change
drought_clim = drought_reg.groupby([drought_reg.index.get_level_values("scenario"),
                                    drought_reg.index.get_level_values("date").month]).mean()
drought_change = 100 * (drought_clim.loc[SCENARIO] / drought_clim.loc["historical"] - 1)
drought_change = drought_change[rows_order]

## Plot

In [ ]:
months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
ylabels = [r.replace(" ", "<br>") for r in rows_order]

melt_scale    = [(0.0, NEG_COLOR), (1.0, "white")]   # -100% (dark) -> 0% (white)
drought_scale = [(0.0, "white"), (1.0, POS_COLOR)]   # 0% (white) -> +150% (dark)
DROUGHT_MAX   = 150

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=[f"a) Glacier melt change ({scenario_name})",
                                    f"b) Drought area change ({scenario_name})"])

# glacier melt change
z = melt_change.T.values
fig.add_trace(go.Heatmap(
    z=z, x=months, y=ylabels, coloraxis="coloraxis",
    text=np.round(z).astype(int).astype(str), texttemplate="%{text}", textfont=dict(size=11),
    xgap=1, ygap=1, hovertemplate="%{y}, %{x}: %{z:.0f}%<extra></extra>",
), row=1, col=1)

# drought area change
z = drought_change.T.values
fig.add_trace(go.Heatmap(
    z=z, x=months, y=ylabels, coloraxis="coloraxis2",
    text=np.round(z).astype(int).astype(str), texttemplate="%{text}", textfont=dict(size=11),
    xgap=1, ygap=1, hovertemplate="%{y}, %{x}: %{z:.0f}%<extra></extra>",
), row=2, col=1)

fig.update_yaxes(autorange="reversed", showgrid=False, ticks="outside", ticklen=5, tickfont=dict(size=12), ticklabelstandoff=4)
fig.update_xaxes(showgrid=False, ticks="outside", ticklen=5, tickfont=dict(size=12), tickmode="array", tickvals=months, ticklabelstandoff=4)
fig.update_xaxes(ticks="", row=1, col=1)  # month ticks only on the bottom panel

fig.update_layout(**base_layout(
    width=800, height=750,
    margin=dict(l=100, r=20, b=50, t=40),
    plot_bgcolor="#d9d9d9",
    coloraxis=dict(colorscale=melt_scale, cmin=-100, cmax=0,
                   colorbar=dict(x=1.01, y=1.0, yanchor="top", len=0.46, thickness=20, outlinewidth=0.5, outlinecolor="black",
                                 ticksuffix="%", ticks="outside")),
    coloraxis2=dict(colorscale=drought_scale, cmin=0, cmax=DROUGHT_MAX,
                    colorbar=dict(x=1.01, y=0.0, yanchor="bottom", len=0.46, thickness=20, outlinewidth=0.5, outlinecolor="black",
                                  ticksuffix="%", ticks="outside",
                                  tickvals=[0, 50, 100, 150], ticktext=["0%", "50%", "100%", "≥150%"])),
))
fig.update_layout(margin=dict(r=80))


save_figure(fig, "figures/fig_whatif_seasonal_regions.svg", width=800, height=750)
fig.show()

## Numbers for the text

In [ ]:
# month of the largest absolute melt loss and of the largest relative loss, per region
summary = pd.DataFrame({
    "peak_melt_month":         melt_hist.idxmax(),
    "max_abs_loss_month":      (melt_whatif - melt_hist).idxmin(),
    "max_rel_loss_month":      melt_change.idxmin(),
    "JJA_melt_change_%":       100 * (melt_whatif.loc[[6, 7, 8]].sum() / melt_hist.loc[[6, 7, 8]].sum() - 1),
    "DJF_share_hist_%":        100 * melt_hist.loc[[12, 1, 2]].sum() / melt_hist.sum(),
    "DJF_share_whatif_%":      100 * melt_whatif.loc[[12, 1, 2]].sum() / melt_whatif.sum(),
    "max_drought_incr_month":  drought_change.idxmax(),
    "max_drought_incr_%":      drought_change.max(),
}).loc[rows_order]
summary.round(1)